In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from sklearn.metrics import log_loss
from supabase import create_client
from dotenv import load_dotenv
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
model = joblib.load('xg_model.pkl')

In [ ]:
def fetch_all(table):
    rows = []
    chunk = 1000
    offset = 0
    while True:
        result = supabase.table(table).select("*").range(offset, offset + chunk - 1).execute()
        rows.extend(result.data)
        if len(result.data) < chunk:
            break
        offset += chunk
    return pd.DataFrame(rows)

shots   = fetch_all("shots")
matches = fetch_all("matches")

In [ ]:
shots1 = shots.copy()
matches1 = matches.copy()

In [ ]:
shots = shots1.copy()
matches = matches1.copy()

In [ ]:
shots = shots[shots['result'] != 'OwnGoal'].copy()
shots = shots[shots['situation'] != 'Penalty'].copy()

shots['understat_xg'] = shots['understat_xg'].astype(float)
shots['x'] = shots['x'].astype(float)
shots['y'] = shots['y'].astype(float)
shots['minute'] = shots['minute'].astype(int)
shots['player_assisted'] = shots['player_assisted'].str.replace('&#039;', "'", regex=False)
shots['is_assisted'] = shots['player_assisted'].notnull().astype(int)
shots = shots.drop(columns=['player_assisted', 'model_xg', 'player', 'player_id', 'h_team', 'a_team', 'date'])

noise_actions = ['SubstitutionOn', 'SubstitutionOff', 'FormationChange', 'Start', 'End']
shots['last_action'] = shots['last_action'].replace(noise_actions, 'Other')
shots = pd.get_dummies(shots, columns=['last_action', 'situation', 'shot_type', 'league', 'h_a'])

GOAL_X, GOAL_Y = 1.0, 0.5
shots['distance']    = np.sqrt((shots['x'] - GOAL_X)**2 + (shots['y'] - GOAL_Y)**2)
shots['angle']       = np.arctan2(abs(shots['y'] - GOAL_Y), abs(shots['x'] - GOAL_X))
shots['off_central'] = abs(shots['y'] - 0.5).astype(float)

drop_cols = ['id', 'result', 'understat_xg', 'season', 'match_id']
feature_cols = [c for c in shots.columns if c not in drop_cols]

model_features = model.calibrated_classifiers_[0].base_estimator.get_booster().feature_names
shots_features = shots[feature_cols].reindex(columns=model_features, fill_value=0)

shots['model_xg'] = model.predict_proba(shots_features)[:, 1]

In [ ]:
match_xg = shots.groupby('match_id').apply(lambda g: pd.Series({
    'home_model_xg':      g[g['h_a_h'] == 1]['model_xg'].sum(),
    'away_model_xg':      g[g['h_a_a'] == 1]['model_xg'].sum(),
    'home_understat_xg':  g[g['h_a_h'] == 1]['understat_xg'].sum(),
    'away_understat_xg':  g[g['h_a_a'] == 1]['understat_xg'].sum(),
})).reset_index()

matches['id'] = matches['id'].astype(str)
match_xg['match_id'] = match_xg['match_id'].astype(str)
df = matches.merge(match_xg, left_on='id', right_on='match_id')
df = df.dropna(subset=['home_goals', 'away_goals'])
df['home_goals'] = df['home_goals'].astype(int)
df['away_goals'] = df['away_goals'].astype(int)
print(df.shape)
df.head()

In [ ]:
def match_probs(home_xg, away_xg, max_goals=10):
    home_probs = poisson.pmf(np.arange(max_goals), home_xg)
    away_probs = poisson.pmf(np.arange(max_goals), away_xg)
    matrix   = np.outer(home_probs, away_probs)
    home_win = np.tril(matrix, -1).sum()
    draw     = np.diag(matrix).sum()
    away_win = np.triu(matrix, 1).sum()
    return home_win, draw, away_win

df[['model_home_win', 'model_draw', 'model_away_win']] = df.apply(
    lambda r: match_probs(r['home_model_xg'], r['away_model_xg']), axis=1, result_type='expand'
)
df[['understat_home_win', 'understat_draw', 'understat_away_win']] = df.apply(
    lambda r: match_probs(r['home_understat_xg'], r['away_understat_xg']), axis=1, result_type='expand'
)

In [ ]:
match_xg_lists = shots.groupby('match_id').apply(lambda g: pd.Series({
    'home_model_xg_list': list(g[g['h_a_h'] == 1]['model_xg']),
    'away_model_xg_list': list(g[g['h_a_a'] == 1]['model_xg']),
})).reset_index()

match_xg_lists['match_id'] = match_xg_lists['match_id'].astype(str)
df = df.merge(match_xg_lists, left_on='id', right_on='match_id', suffixes=('', '_dup'))

def poisson_binomial_pmf(probs, max_goals=15):
    pmf = np.zeros(max_goals)
    pmf[0] = 1.0
    for p in probs:
        new_pmf = np.zeros(max_goals)
        for k in range(max_goals):
            new_pmf[k] += pmf[k] * (1 - p)
            if k + 1 < max_goals:
                new_pmf[k + 1] += pmf[k] * p
        pmf = new_pmf
    return pmf

def match_probs_pb(home_xg_list, away_xg_list, max_goals=15):
    home_dist = poisson_binomial_pmf(home_xg_list, max_goals)
    away_dist = poisson_binomial_pmf(away_xg_list, max_goals)
    matrix   = np.outer(home_dist, away_dist)
    home_win = np.tril(matrix, -1).sum()
    draw     = np.diag(matrix).sum()
    away_win = np.triu(matrix, 1).sum()
    return home_win, draw, away_win

df[['pb_home_win', 'pb_draw', 'pb_away_win']] = df.apply(
    lambda r: match_probs_pb(r['home_model_xg_list'], r['away_model_xg_list']),
    axis=1, result_type='expand'
)

In [ ]:
df['actual_result'] = np.where(df['home_goals'] > df['away_goals'], 'H',
                      np.where(df['home_goals'] < df['away_goals'], 'A', 'D'))

result_map = {'H': [1,0,0], 'D': [0,1,0], 'A': [0,0,1]}
y_true = np.array([result_map[r] for r in df['actual_result']])

def get_prediction(row, prefix):
    probs = {
        'H': row[f'{prefix}_home_win'],
        'D': row[f'{prefix}_draw'],
        'A': row[f'{prefix}_away_win'],
    }
    return max(probs, key=probs.get)

df['model_pred']     = df.apply(lambda r: get_prediction(r, 'model'), axis=1)
df['understat_pred'] = df.apply(lambda r: get_prediction(r, 'understat'), axis=1)
df['pb_pred']        = df.apply(lambda r: get_prediction(r, 'pb'), axis=1)

model_ll     = log_loss(y_true, df[['model_home_win', 'model_draw', 'model_away_win']].values)
understat_ll = log_loss(y_true, df[['understat_home_win', 'understat_draw', 'understat_away_win']].values)
pb_ll        = log_loss(y_true, df[['pb_home_win', 'pb_draw', 'pb_away_win']].values)

model_acc     = (df['model_pred'] == df['actual_result']).mean()
understat_acc = (df['understat_pred'] == df['actual_result']).mean()
pb_acc        = (df['pb_pred'] == df['actual_result']).mean()

print(f"Poisson-Binomial log-loss: {pb_ll:.4f}  accuracy: {pb_acc:.3f}")
print(f"Poisson          log-loss: {model_ll:.4f}  accuracy: {model_acc:.3f}")
print(f"Understat        log-loss: {understat_ll:.4f}  accuracy: {understat_acc:.3f}")

In [ ]:
match = df[df['id'] == '7490'].iloc[0]
print(f"Home team: {match['home_team']}")
print(f"Away team: {match['away_team']}")
print(f"Actual result: {match['home_goals']}-{match['away_goals']}")
print()
print(f"Your model:  Home {match['pb_home_win']:.3f}  Draw {match['pb_draw']:.3f}  Away {match['pb_away_win']:.3f}")
print(f"Understat:   Home {match['understat_home_win']:.3f}  Draw {match['understat_draw']:.3f}  Away {match['understat_away_win']:.3f}")